# Spatial analysis and islet inflammation

Inspect a whole pancreatic section, measure cell neighborhoods from native centroids, and compare measured islet inflammation across donors and disease groups.

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/03_spatial_analysis.ipynb).

## 1. Select a donor

The native-cell graph connects cells within a section. The islet graph has a separate fit for each donor.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
from phenotype_signatures import (cell_subtypes, phenotype_colors, phenotype_key, names_in_order, subtype_display_name)
release = validate_release(data)

DONOR = "115"                      # Any donor listed in the manifest
DISEASE = "ND"                     # ND, AAb+, or T1D
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
inks = phenotype_colors(counts, "Pancreas")

## 2. Spatial visualization

The most specific recorded cell assignments and islet compartments are shown at their native centroids using a uniform display sample. The CD99 endocrine group with no measurable hormone above the typing floor and the competing-hormone group retain separate labels.

In [ ]:
# Native coordinates, µm. Draw a uniform subset of centroids to keep whole sections legible.
donor_path = data / "pancreas" / f"{DONOR}.h5ad"
donor = sc.read_h5ad(donor_path,backed="r")
assert donor.n_obs == manifest.set_index("donor").at[DONOR,"cells"]
rng = np.random.default_rng(19)
ix = np.sort(rng.choice(donor.n_obs,min(donor.n_obs,60000),replace=False))
labels = cell_subtypes(donor.obs, "Pancreas")
obs = donor.obs.iloc[ix][["islet_region"]].copy()
obs["subtype"] = labels.iloc[ix].cat.remove_unused_categories()
obs["islet_region"] = obs.islet_region.cat.remove_unused_categories()
xy = np.asarray(donor.obsm["spatial"])[ix]
section_sample = ad.AnnData(obs=obs, obsm={"spatial":xy})
zones = {"outside":"#c8c6c1","core":"#126891","peri":"#d78939","multiple peri-islet bands":"#6e4c88"}
shown = names_in_order(obs.subtype.cat.categories, "Pancreas")
legend_height = .24 * np.ceil(len(shown) / 3) + .4
fig = plt.figure(figsize=(13, 5.8 + legend_height), layout="constrained")
grid = fig.add_gridspec(2, 2, height_ratios=[5,legend_height])
for j,(key,palette,title) in enumerate([("subtype",inks,"Cell subtype"),("islet_region",zones,"Islet compartment")]):
    ax = fig.add_subplot(grid[0,j])
    sc.pl.embedding(section_sample,basis="spatial",color=key,ax=ax,show=False,
                    size=3,palette=palette,legend_loc="none",title=title)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    if key == "islet_region":
        handles = [Line2D([], [], marker="o", ms=4, ls="", color=zones[v], label=v) for v in obs.islet_region.cat.categories]
        ax.legend(handles=handles,loc="upper center",bbox_to_anchor=(.5,0),ncol=2,frameon=False,fontsize=8)
phenotype_key(fig.add_subplot(grid[1,:]), shown, inks, columns=3)
fig.suptitle(f"Donor {DONOR} · {donor.n_obs:,} admitted cells · {len(ix):,} uniformly sampled centroids")
plt.show()

## 3. Spatial neighborhood enrichment

Each row is a focal phenotype and each column is a neighbor phenotype. The graph uses every admitted centroid in the selected section. The heatmap shows descriptive enrichment relative to the section abundance. Gray cells have no observed edges or an undefined expectation.

In [ ]:
from scipy.spatial import cKDTree

# Ten nearest other cells, computed within this section from all admitted native centroids.
xy = np.asarray(donor.obsm["spatial"], dtype=np.float64)
labels = cell_subtypes(donor.obs, "Pancreas").cat.remove_unused_categories()
groups = labels.cat.categories
codes = labels.cat.codes.to_numpy(dtype=np.int64)
tree = cKDTree(xy)
k = min(10, len(xy)-1)
pairs = np.zeros((len(groups),len(groups)),dtype=np.int64)
for start in range(0,len(xy),50000):
    stop = min(start+50000,len(xy))
    _, neighbors = tree.query(xy[start:stop],k=k+1, workers=2)
    centers = np.arange(start,stop)[:,None]
    rank = np.cumsum(neighbors!=centers,axis=1)
    take = (neighbors!=centers) & (rank<=k)
    edges = codes[start:stop,None]*len(groups) + codes[neighbors]
    pairs += np.bincount(edges[take],minlength=len(groups)**2).reshape(pairs.shape)
assert pairs.sum() == len(xy)*k
# Enrichment relative to random mixing at the observed section-wide abundance.
# A cell cannot be its own neighbor; use the finite-population expectation.
abundance = np.bincount(codes,minlength=len(groups))
null = (abundance[None,:] - np.eye(len(groups))) / (len(xy)-1)
pobs = pairs / pairs.sum(axis=1,keepdims=True)
ratio = np.log2(np.divide(pobs,null,out=np.full_like(pobs,np.nan),where=(pobs>0)&(null>0)))
order = [groups.get_loc(t) for t in names_in_order(groups, "Pancreas")]
mat = ratio[np.ix_(order,order)]; names=[subtype_display_name(label) for label in groups[order]]
side = max(6.4, .25 * len(names))
fig, ax = plt.subplots(figsize=(side + 4.3, side + 2.2), layout="constrained")
lim=max(1,float(np.nanpercentile(np.abs(mat),95)))
cmap=plt.get_cmap("RdBu_r").copy();cmap.set_bad("#dadada")
im=ax.imshow(mat,cmap=cmap,norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim),aspect="auto")
ax.set(xticks=range(len(names)),yticks=range(len(names)),xticklabels=names,yticklabels=names,
       title=f"Cell neighborhoods · donor {DONOR}",xlabel="Neighbor identity",ylabel="Focal cell identity")
ax.tick_params(axis="both", labelsize=8)
plt.setp(ax.get_xticklabels(),rotation=65,ha="right",rotation_mode="anchor")
fig.colorbar(im,ax=ax,label=r"$\log_2$(observed / expected)",shrink=.72)
plt.show()

## 4. The selected donor’s islet trajectory

Each point is one islet. The graph uses this donor’s core and peri-islet cell composition. Lines connect the donor’s communities along the PAGA tree. The star identifies the fitted islet with the lowest measured combined immune density. Where several share that minimum, the nonimmune reference distance and then the islet ID settle the tie.

Color gives immune cells/mm² on a scale shared across donors. The middle panel orders this donor’s assessed islets by the actual combined density and shows the separate core and rim responses. The right panel shows those responses along diffusion pseudotime for graph-fitted islets. Both panels mark medians in successive intervals. A decrease along the graph is visible in the measurements. Diffusion pseudotime orders cross-sectional islet compositions. The positions in each donor’s graph have their own orientation.

In [ ]:
from matplotlib.ticker import FuncFormatter

islets = sc.read_h5ad(data / "pancreas_islets.h5ad")
density = "combined_immune_per_mm2"
rank = "pseudotime_rank"
if islets.uns.get("inflammation_order_version") != "native-core-peri20-shared-density-v2":
    raise ValueError("Select the release with a shared inflammation scale across all donors.")
assessed = islets.obs.loc[islets.obs.density_eligible].copy()
np.testing.assert_allclose(assessed[density], 1e6 * (assessed.n_core_immune + assessed.n_peri_immune)
                           / (assessed.core_area_um2 + assessed.peri_area_um2))
peak = assessed[density].max()
np.testing.assert_allclose(assessed.pseudotime, assessed[density] / peak)
assert assessed.sort_values(density)[rank].is_monotonic_increasing
ticks = [v for v in (0, 250, 1000, 2500, 5000, 10000, 20000, 40000) if v <= peak]
def density_axis(ax, direction, limit, values):
    setter = ax.set_xscale if direction == "x" else ax.set_yscale
    setter("function", functions=(lambda x: np.sqrt(np.clip(x, 0, None)), lambda x: np.square(x)))
    axis = ax.xaxis if direction == "x" else ax.yaxis
    axis.set_ticks(values)
    axis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:,.0f}"))
    (ax.set_xlim if direction == "x" else ax.set_ylim)(0, limit * 1.05)

from islet_trajectory_plots import load_trajectories, plot_donor_trajectory, plot_trajectory_atlas
trajectories = load_trajectories(data)
plot_donor_trajectory(trajectories, DONOR)
plt.show()


## 5. Individual islet trajectories for all 25 donors

Each panel is a separately fitted graph rooted at low inflammation in that donor. Counts give fitted and native islets. The density color scale is common across disease groups. The [donor trajectory atlas](../rendered/donor_islet_trajectories.pdf) gives each donor a full page with their graph and changes in inflammation along it.

In [ ]:
plot_trajectory_atlas(trajectories)
plt.show()

## 6. Comparable inflammation ranges

Absolute immune cells/mm² preserve the observed endpoints across ND, AAb+ and T1D. Thin segments give each donor’s minimum and maximum, thick segments give the interquartile range, and dots mark the median. The axis uses square-root spacing with ticks labeled in cells/mm². The table summarizes assessed islets pooled by disease group.

In [ ]:
donor_ranges = assessed.groupby(["disease","donor"], observed=True)[density].quantile(
    [0,.25,.5,.75,1]).unstack().reindex(columns=[0,.25,.5,.75,1])
donor_ranges.columns = ["Minimum","Q1","Median","Q3","Maximum"]
donor_ranges = donor_ranges.reset_index()
fig, axes = plt.subplots(1, len(group_order), figsize=(13,5.1), sharex=True, layout="constrained")
for ax, group in zip(axes, group_order):
    z = donor_ranges.loc[donor_ranges.disease.eq(group)].sort_values("Median")
    y = np.arange(len(z))
    ax.hlines(y, z.Minimum, z.Maximum, color=group_ink[group], lw=1.1)
    ax.hlines(y, z.Q1, z.Q3, color=group_ink[group], lw=5)
    ax.scatter(z.Median, y, facecolor="white", edgecolor="#182c3b", s=26, zorder=3)
    ax.scatter(z.Maximum, y, color=group_ink[group], s=14)
    density_axis(ax, "x", peak, [v for v in (0,1000,5000,10000) if v<=peak])
    ax.set(yticks=y, yticklabels=z.donor.astype(str), ylim=(-.8,max(1,len(y))-.2), title=group)
    ax.grid(axis="x", alpha=.2)
fig.suptitle("Observed islet inflammation ranges by donor")
fig.supxlabel("Combined immune density (cells/mm²)")
plt.show()
group_summary = pd.read_csv(data / "islet_inflammation_by_disease.csv").set_index("disease").reindex(group_order)
display(group_summary.round(3))
display(donor_ranges.set_index(["disease","donor"]).round(2))

## 7. How does the original composition graph relate to inflammation?

The original within-donor diffusion pseudotime (DPT) is rooted using nonimmune composition and reflects several cell populations. Compare it against measured immune-cell density for the islets with both estimates. Each point summarizes one donor. The table also reports islets at zero measured burden and the number eligible for each ordering.

In [ ]:
res = []
for (group, did), frame in islets.obs.groupby(["disease","donor"],observed=True):
    z = frame.dropna(subset=["composition_dpt_rank", density])
    rho = (z.composition_dpt_rank.corr(z[density], method="spearman")
           if z.composition_dpt_rank.nunique()>1 and z[density].nunique()>1 else np.nan)
    res.append({"disease":group, "donor":did, "native_islets":len(frame),
                "inflammation_ordered":frame[rank].count(), "zero_density":frame[density].eq(0).sum(),
                "composition_DPT_measured":len(z), "rho":rho})
overview = pd.DataFrame(res).set_index("donor")
fig, ax = plt.subplots(figsize=(7,3.8))
for i,g in enumerate(group_order):
    y=overview.loc[overview.disease.eq(g),"rho"].dropna()
    ax.scatter(i+np.linspace(-.15,.15,len(y)),y,color=group_ink[g],s=30)
    if len(y):
        ax.hlines(y.median(),i-.23,i+.23,color="#333")
ax.axhline(0,color="#777",lw=.7)
ax.set(xticks=range(3),xticklabels=group_order,ylim=(-1,1),ylabel="Within-donor Spearman ρ",
       title="Composition DPT vs. measured core + peri-islet immune density")
fig.tight_layout()
plt.show()
display(overview)